In [1]:
!pip install langchain-openai langchain-community

In [2]:
!pip install -qU langchain-huggingface sentence_transformers huggingface_hub



In [1]:
from dotenv import load_dotenv 
load_dotenv()

True

In [4]:
import os
os.environ.get("LANGSMITH_API_KEY")

In [4]:
import os
base_url0 = os.getenv("BASE_URL")
openai_api_key = os.getenv("OPENAI_API_KEY")
import getpass
if not openai_api_key:
    openai_api_key=getpass.getpass()
if not base_url0:
    base_url0 = getpass.getpass()

In [5]:
from langchain_community.document_loaders import TextLoader

documento =TextLoader("documentos/GTB_gold_Nov23.txt", encoding="utf-8").load()

c:\source\engenharia_ia\LangChain Avançada\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
documento

[Document(metadata={'source': 'documentos/GTB_gold_Nov23.txt'}, page_content='\n1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos  por,  para,  ou  emitidos  para  a \nMas

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,chunk_overlap=100
)

pedaços = splitter.split_documents(documento)

In [8]:
len(pedaços)

5

In [9]:
pedaços[2]

Document(metadata={'source': 'documentos/GTB_gold_Nov23.txt'}, page_content='uma perda ou pedido de serviços forem efetuados, se você intencionalmente ocultar ou fizer \ninterpretação  errônea  de  qualquer  fato  material  ou  circunstância,  ou  fornecer  informação  fraudulenta \nrelativa  aos  planos  de  seguro  ou  outros  serviços  aqui  descritos  para:  A  Mastercard  International,  a \nEmpresa de Seguros, a instituição financeira que emitiu a Conta do cartão ou qualquer outra empresa \nque estiver prestando serviços e/ou administração em nome destes programas. \n \nPara  dar  entrada  em  uma  ocorrência/sinistro  ou  para  obter  mais  informações  sobre  qualquer um \ndesses serviços, ligue para o número gratuito do Mastercard Global Service™ específico para o seu país, \nou ligue a cobrar para os Estados Unidos no número 1-636-722-8881 (Português). \n “cartão” refere-se ao cartão Mastercard Gold. \n \n“portador de cartão”, “você”, e “seu” referem-se a um  portador do cart

In [28]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

modelo = ChatOpenAI(
   model = "local-model",
   base_url=base_url0,
   api_key=openai_api_key,
   temperature=0.8,
   max_tokens=250
)


embedings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'} 
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5279.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
pedaços[0].page_content

'1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos  por,  para,  ou  emitidos  para  a \nMastercard. \n \nNome  do Representante: MASTERCARD DO BRASIL LTDA.  CNPJ 01.248.2

In [13]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS


# 1. Carrega o seu arquivo TXT (mude o nome para o seu arquivo real)
loader = TextLoader("documentos/GTB_gold_Nov23.txt", encoding="utf-8")
documentos = loader.load()

# 2. Divide em pedaços para a IA não se perder
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
pedaços = text_splitter.split_documents(documentos)

# 3. Cria o banco de dados de vetores (isso roda no seu PC usando o MiniLM)
db = FAISS.from_documents(pedaços, embedings)

print(f"Sucesso! {len(pedaços)} pedaços de texto foram indexados.")

Sucesso! 1 pedaços de texto foram indexados.


In [15]:
from langchain_community.vectorstores import InMemoryVectorStore

embeddings_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(documents= pedaços, embedding=embeddings_model)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8581.23it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
retriver = vectorstore.as_retriever(search_kwargs={"k":2})

In [17]:
retriver.invoke("seguro viagem")

[Document(id='14aa3e97-6712-44bb-b310-50bf15a5749e', metadata={'source': 'documentos/GTB_gold_Nov23.txt'}, page_content='1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos 

In [18]:
query = "como devo proceder caso tenha um item comprado roubado"

query_embed = embeddings_model.embed_query(query)
query

'como devo proceder caso tenha um item comprado roubado'

In [19]:
similar_chunks = retriver.invoke(query)
similar_chunks

[Document(id='14aa3e97-6712-44bb-b310-50bf15a5749e', metadata={'source': 'documentos/GTB_gold_Nov23.txt'}, page_content='1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos 

In [20]:
similiar_text= [ pedaço.page_content for pedaço in similar_chunks]
similar_chunks

[Document(id='14aa3e97-6712-44bb-b310-50bf15a5749e', metadata={'source': 'documentos/GTB_gold_Nov23.txt'}, page_content='1\n1 \nVersão: novembro 2023 \n2021 \n \n \n \nPrograma de Cartão da Edição Mastercard Gold  \nGuia de Benefícios \n Informações importantes. Leia e guarde as informações. \n \nEste Guia de Benefícios contém informações detalhadas sobre serviços abrangentes de viagem, seguros \ne assistência aos quais você terá acesso como portador de cartão preferencial. Esses benefícios e serviços \nestão em vigor para portadores do cartão Mastercard Gold elegível a partir de 1 de Novembro de  2023. \nEste Guia substitui qualquer  guia ou comunicação de programa que você recebeu anteriormente. \n \nAs informações contidas neste documento são apresentadas somente com propósito informativo. Não \npretendem  ser  uma  descrição  completa  de  todos  os  termos,  condições,  limitações, exclusões  ou  outras \ndisposições  de  qualquer  programa  ou  benefícios  de  seguro  fornecidos 

In [21]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","Responda usando exclusivamente os conteúdo fornecido. \n\nContexto:\n{Contexto}"),
        ("human","{query}")
    ]
)

In [29]:
modelo.invoke(query)

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 250, 'prompt_tokens': 20, 'total_tokens': 270, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen/qwen3.5-9b', 'system_fingerprint': 'qwen/qwen3.5-9b', 'id': 'chatcmpl-rodkn021htj3suze22fj6r', 'finish_reason': 'length', 'logprobs': None}, id='lc_run--019d0cc3-bd0c-7070-93c9-2b10b028f6bb-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 250, 'total_tokens': 270, 'input_token_details': {}, 'output_token_details': {}})